### 1. The Perceptron & Deep Learning Frameworks
A Perceptron is the simplest form of an artificial neural network used for binary classification. It computes a linear combination of inputs, weights, and a bias, then applies an activation function:
`f(x) = Σ(w_i * x_i) + b`

**Types of Perceptrons:**
| Type | Description |
| :--- | :--- |
| **Single Layer** | Works only for linearly separable problems (e.g., AND, OR gates). |
| **Multi-Layer** | Can solve complex (non-linear) problems (e.g., XOR gate). |


### 2. Step 1: Import Necessary Libraries
We import `torch` for core PyTorch functionality, `torch.nn` for neural network layers, and `torch.optim` for optimization algorithms like Adam or SGD.


In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

### 3. Step 2: Create and Split Synthetic Dataset
We generate a synthetic binary classification dataset. Notice the use of elaborative variable names to clarify what the arrays represent.


In [3]:
X, y = make_classification(
    n_samples=1000,
    n_features=2,
    n_informative=2,
    n_redundant=0,
    n_classes=2,
    random_state=42
)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

### 4. Step 3: Standardize the Dataset
Standardization helps neural networks converge faster and enhances accuracy by ensuring all features have a mean of 0 and a standard deviation of 1.
*Note: We fit the scaler ONLY on the training data to prevent data leakage!*


In [5]:
feature_scaler = StandardScaler()

# Fit and transform the training features
X_train_scaled = feature_scaler.fit_transform(X_train)

# ONLY transform the testing features using the learned parameters
X_test_scaled = feature_scaler.transform(X_test)

### 5. Step 4: Convert Data to PyTorch Tensors
PyTorch requires data to be in a specific structure called a **Tensor**, which supports automatic differentiation for backpropagation.

**Common Tensor Data Types in PyTorch:**
| dtype | Description |
| :--- | :--- |
| `torch.float32` | Standard float (most common for neural network inputs) |
| `torch.int64` | 64-bit integer (default for integers/labels) |

We convert our NumPy arrays to PyTorch Tensors. We also reshape the targets into column vectors `(-1, 1)` so their matrix shape aligns with the network's output.


In [7]:
# Convert scaled features to float32 tensors
X_train_scaled_tensor = torch.tensor(X_train_scaled, dtype=torch.float32)
X_test_scaled_tensor = torch.tensor(X_test_scaled, dtype=torch.float32)

# Convert labels and reshape them from flat arrays (N,) to column vectors (N, 1)
y_train_tensor = torch.tensor(y_train, dtype=torch.float32).reshape(-1, 1)
y_test_tensor = torch.tensor(y_test, dtype=torch.float32).reshape(-1, 1)

### 6. Verify Tensor Shapes
It is crucial to verify your matrix dimensions before feeding them into a neural network.


In [8]:
print("Training Features Shape:", X_train_scaled_tensor.shape)
print("Training Labels Shape:", y_train_tensor.shape)
print("Testing Features Shape:", X_test_scaled_tensor.shape)
print("Testing Labels Shape:", y_test_tensor.shape)


Training Features Shape: torch.Size([800, 2])
Training Labels Shape: torch.Size([800, 1])
Testing Features Shape: torch.Size([200, 2])
Testing Labels Shape: torch.Size([200, 1])


### 7. Step 5: Building a Neural Network (PyTorch Layers & Activations)
In PyTorch, networks inherit from `nn.Module`. 

**Typical Layer & Activation Combinations:**
| Problem Type | Final Layer | Activation |
| :--- | :--- | :--- |
| Binary Classification | `nn.Linear(..., 1)` | Sigmoid |
| Multi-Class Classification | `nn.Linear(..., num_classes)` | Softmax |
| Regression | `nn.Linear(..., 1)` | None |
| Hidden Layer | `nn.Linear(...)` | ReLU |

We will define a `BinaryClassifier` class representing our single-layer perceptron.


In [15]:
class Perceptron(nn.Module):
    def __init__(self, input_features):
        # Call the parent class constructor
        super(Perceptron, self).__init__()
        
        # Define a fully connected layer (Dense Layer): Inputs -> 1 Output Neuron
        self.fully_connected_layer = nn.Linear(in_features=input_features, out_features=1)
        
    def forward(self, input_data):
        # 1. Pass data through the linear weights and bias
        linear_output = self.fully_connected_layer(input_data)
        
        # 2. Pass the linear result through a Sigmoid activation (outputs between 0 and 1)
        activated_output = torch.sigmoid(linear_output)
        
        return activated_output

# Instantiate the model with 2 input features
model = Perceptron(input_features=2)

### 8. Inspecting Model Parameters
Let's print the model and its initialized weights and bias. PyTorch initializes these randomly.


In [16]:
print("Model Architecture:")
print(model)

print("\nInitialized Parameters (Weights and Bias):")
for name, param in model.named_parameters():
    print(f"{name}: {param.data}")

Model Architecture:
Perceptron(
  (fully_connected_layer): Linear(in_features=2, out_features=1, bias=True)
)

Initialized Parameters (Weights and Bias):
fully_connected_layer.weight: tensor([[-0.2102, -0.6088]])
fully_connected_layer.bias: tensor([0.5561])


### 9. Alternative: Using `nn.Sequential`
If you don't need a complex forward pass, you can stack layers instantly using `nn.Sequential`. This code works exactly the same as we our Perceptron class model works. But we continue using Percentron Model onwards.

In [17]:
sequential_model = nn.Sequential(
    nn.Linear(in_features=2, out_features=1),
    nn.Sigmoid()
)
print(sequential_model)

Sequential(
  (0): Linear(in_features=2, out_features=1, bias=True)
  (1): Sigmoid()
)


### 10. Step 6: Define Loss Function and Optimizer

**Common PyTorch Loss Functions:**
| Loss Function | Syntax | Used For |
| :--- | :--- | :--- |
| Mean Squared Error | `nn.MSELoss()` | Regression |
| Binary Cross Entropy | `nn.BCELoss()` | Binary Classification (0 or 1 targets) |

**Common PyTorch Optimizers:**
| Optimizer | Syntax | Key Idea |
| :--- | :--- | :--- |
| SGD | `optim.SGD()` | Stochastic Gradient Descent (basic) |
| Adam | `optim.Adam()` | Adaptive learning rate (most commonly used) |

We initialize BCE Loss and the Adam optimizer.


In [19]:
# Measures how far predicted probabilities are from true 0 or 1 labels
criterion = nn.BCELoss()

# Updates model weights using computed gradients to minimize the loss
optimizer = optim.Adam(params=model.parameters(), lr=0.01)

### 11. Step 7: Train the Model
The core deep learning training loop consists of 4 repetitive steps per epoch:
1. **Forward Pass:** Ask the model to make predictions on the training data.
2. **Compute Loss:** Calculate how wrong the predictions were.
3. **Zero Gradients & Backward Pass:** Clear old memory, then calculate the derivatives (gradients) using the chain rule.
4. **Step Optimizer:** Adjust the weights in the direction that reduces the error.


In [23]:
total_epochs = 100

print("Starting Training Loop...")
for epoch in range(total_epochs):
    
    # 1. Forward Pass: Get probabilities from the model
    predicted_probabilities = model(X_train_scaled_tensor)
    
    # 2. Compute Loss
    current_loss = criterion(predicted_probabilities, y_train_tensor)
    
    # 3. Reset old gradients to zero (PyTorch accumulates them by default)
    optimizer.zero_grad()
    
    # 4. Backpropagation: Calculate gradients
    current_loss.backward()
    
    # 5. Update weights
    optimizer.step()
    
    # Print progress every 10 epochs
    if (epoch + 1) % 10 == 0:
        print(f"Epoch [{epoch + 1}/{total_epochs}], Loss: {current_loss.item():.4f}")


Starting Training Loop...
Epoch [10/100], Loss: 0.7083
Epoch [20/100], Loss: 0.6767
Epoch [30/100], Loss: 0.6482
Epoch [40/100], Loss: 0.6226
Epoch [50/100], Loss: 0.5995
Epoch [60/100], Loss: 0.5786
Epoch [70/100], Loss: 0.5598
Epoch [80/100], Loss: 0.5427
Epoch [90/100], Loss: 0.5273
Epoch [100/100], Loss: 0.5134


### 12. Step 8: Model Evaluation
After training, we evaluate on the test set. We use `torch.no_grad()` to tell PyTorch to stop tracking gradients, which saves memory and computation speed during inference.


In [ ]:
with torch.no_grad():
    # Get raw probabilities on unseen test data
    y_pred_prob_tensor = model(X_test_scaled_tensor)
    
    # Convert probabilities to binary classes (Threshold = 0.5)
    y_pred_tensor = (y_pred_prob_tensor > 0.5).float()
    
    # Calculate Accuracy
    correct_predictions = (y_pred_tensor == y_test_tensor).sum().item()
    total_samples = y_test_tensor.size(0)
    
    test_accuracy = correct_predictions / total_samples
    print(f"Final Test Accuracy: {test_accuracy * 100:.2f}%")

Final Test Accuracy: 87.50%


### 13. Advanced Metrics using Scikit-Learn
PyTorch integrates easily with scikit-learn for detailed evaluation metrics.


In [26]:
from sklearn.metrics import confusion_matrix, classification_report

# Convert tensors back to numpy arrays for sklearn compatibility
y_pred_np = y_pred_tensor.numpy()
y_test_np = y_test_tensor.numpy()

print("--- Confusion Matrix ---")
print(confusion_matrix(y_test_np, y_pred_np))

print("\n--- Classification Report ---")
print(classification_report(y_test_np, y_pred_np))

--- Confusion Matrix ---
[[89 12]
 [13 86]]

--- Classification Report ---
              precision    recall  f1-score   support

         0.0       0.87      0.88      0.88       101
         1.0       0.88      0.87      0.87        99

    accuracy                           0.88       200
   macro avg       0.88      0.87      0.87       200
weighted avg       0.88      0.88      0.87       200

